<a href="https://colab.research.google.com/github/franciscorafael-dotcom/DL_P2_TP2_G2-F1/blob/main/treino_cnn_resnet50_effnetb0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Treino CNN - ResNet50 e EfficientNet-B0 (P2 · Tema F)
# Fine-tuning de duas CNN pré-treinadas (ImageNet) no FieldVitis, com o `splits.json` do repositório (3 seeds).

In [5]:
# --- Configuração: dados (Drive) + splits (GitHub) ---
from google.colab import drive
drive.mount("/content/drive")

!git clone -q https://github.com/franciscorafael-dotcom/DL_P2_TP2_G2-F1.git
!mkdir -p /content/fieldvitis && unzip -q -o /content/drive/MyDrive/fieldvitis.zip -d /content/fieldvitis/

import json
from pathlib import Path
DATA_DIR = Path("/content/fieldvitis")
REPO_DIR = Path("/content/DL_P2_TP2_G2-F1")

# Verificação: as imagens batem com o splits.json?
sj = json.load(open(REPO_DIR / "data/splits.json"))
CLASSES = sj["classes"]
print("Pastas:      ", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))
print("splits.json: ", CLASSES)
for s in ["0", "1", "2"]:
    sp = sj["seeds"][s]
    faltam = [p for k in sp for p in sp[k] if not (DATA_DIR / p).exists()]
    print(f"seed {s}:", {k: len(v) for k, v in sp.items()}, "| em falta:", len(faltam))

Mounted at /content/drive
Pastas:       ['Black_rot', 'Esca', 'Healthy', 'Leaf_blight']
splits.json:  ['Black_rot', 'Esca', 'Healthy', 'Leaf_blight']
seed 0: {'train': 388, 'val': 84, 'test': 84} | em falta: 0
seed 1: {'train': 388, 'val': 84, 'test': 84} | em falta: 0
seed 2: {'train': 388, 'val': 84, 'test': 84} | em falta: 0


In [6]:
# Configuração e Carregamento das imagens
import time, gc
import numpy as np, pandas as pd
import tensorflow as tf, keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import f1_score, recall_score

IMG_SIZE, BATCH = 224, 32
EPOCHS_HEAD, LR_HEAD = 5, 1e-3     # fase 1: só a cabeça (base congelada)
EPOCHS_FT, LR_FT = 25, 1e-5        # fase 2: fine-tuning da rede toda
PATIENCE = 6                       # early stopping no Macro-F1 de validação
SEEDS = [0, 1, 2]
RUNS_DIR = Path("/content/drive/MyDrive/P2_runs"); RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Keras", keras.__version__, "| backend", keras.backend.backend(),
      "| GPU:", tf.config.list_physical_devices("GPU"))

# Todas as imagens carregadas uma vez: RGB, orientação EXIF corrigida, 224x224
all_paths = sorted({p for s in sj["seeds"].values() for k in ("train", "val", "test") for p in s[k]})
cls_idx = {c: i for i, c in enumerate(CLASSES)}
IMGS, LABELS = {}, {}
for p in all_paths:
    im = ImageOps.exif_transpose(Image.open(DATA_DIR / p)).convert("RGB")
    IMGS[p] = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)
    LABELS[p] = cls_idx[Path(p).parts[0]]
print("Imagens carregadas:", len(IMGS), "| por classe:", np.bincount(list(LABELS.values())))

Keras 3.13.2 | backend tensorflow | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Imagens carregadas: 556 | por classe: [ 39 201 158 158]


In [7]:
# Cada rede espera os píxeis num formato diferente (o mesmo usado no treino do ImageNet)
PREPROCESS = {
    "resnet50": keras.applications.resnet50.preprocess_input,           # converte para BGR e subtrai a média do ImageNet
    "efficientnetb0": keras.applications.efficientnet.preprocess_input,  # não altera: o modelo normaliza por dentro
}

def preparar_dados(seed, nome):
    """Devolve treino/val/teste da seed indicada, já pré-processados para o modelo."""
    sp = sj["seeds"][str(seed)]
    dados = {}
    for split in ["train", "val", "test"]:
        x = np.stack([IMGS[p] for p in sp[split]]).astype("float32")
        y = np.array([LABELS[p] for p in sp[split]])
        dados[split] = (PREPROCESS[nome](x), y, sp[split])
    return dados

In [8]:
def criar_modelo(nome):
    """Base pré-treinada no ImageNet + augmentation + camada final de 4 classes."""
    if nome == "resnet50":
        base = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                           input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    else:
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet",
                                                 input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    base.trainable = False   # começa congelada

    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.2),
    ])  # só atua durante o treino

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inputs)
    x = base(x, training=False)   # BatchNorm fica em modo inferência, mesmo no fine-tuning
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    return keras.Model(inputs, outputs), base

In [14]:
# Treinar e Guardar
def guardar_previsoes(nome, seed, split, caminhos, y, probs):
    """CSV para o avaliador: uma linha por imagem, com a classe verdadeira, a prevista e as 4 probabilidades."""
    df = pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASSES])
    df.insert(0, "y_pred", [CLASSES[i] for i in probs.argmax(axis=1)])
    df.insert(0, "y_true", [CLASSES[i] for i in y])
    df.insert(0, "path", caminhos)
    df.insert(0, "split", split)
    df.insert(0, "seed", seed)
    df.insert(0, "modelo", nome)
    df.to_csv(RUNS_DIR / f"preds_{nome}_seed{seed}_{split}.csv", index=False)

def treinar(nome, seed):
    keras.utils.set_random_seed(seed)   # reprodutibilidade
    dados = preparar_dados(seed, nome)
    x_tr, y_tr, _ = dados["train"]
    x_va, y_va, _ = dados["val"]

    # Pesos por classe: o black rot (poucas imagens) pesa mais na loss
    contagens = np.bincount(y_tr, minlength=len(CLASSES))
    class_weight = {i: len(y_tr) / (len(CLASSES) * n) for i, n in enumerate(contagens)}

    model, base = criar_modelo(nome)

    # Fase 1: treina só a camada final (base congelada)
    model.compile(optimizer=keras.optimizers.Adam(LR_HEAD),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_HEAD,
              batch_size=BATCH, class_weight=class_weight, verbose=2)

    # Fase 2: fine-tuning de toda a rede, com learning rate baixo
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(LR_FT),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                          restore_best_weights=True)
    model.fit(x_tr, y_tr, validation_data=(x_va, y_va), epochs=EPOCHS_FT,
              batch_size=BATCH, class_weight=class_weight, callbacks=[early], verbose=2)

    # Guardar o modelo (só no Drive) e as previsões de val e teste (para o avaliador)
    model.save(RUNS_DIR / f"{nome}_seed{seed}.keras")
    for split in ["val", "test"]:
        x, y, caminhos = dados[split]
        guardar_previsoes(nome, seed, split, caminhos, y, model.predict(x, verbose=0))

    _, val_acc = model.evaluate(x_va, y_va, verbose=0)
    print(f">>> {nome} seed {seed}: accuracy de validação = {val_acc:.3f}")

In [15]:
# ResNet50
for seed in SEEDS:
    print(f"\n===== ResNet50 | seed {seed} =====")
    treinar("resnet50", seed)
    keras.backend.clear_session()   # liberta a memória da GPU entre corridas


===== ResNet50 | seed 0 =====
Epoch 1/5
13/13 - 11s - 811ms/step - accuracy: 0.4175 - loss: 1.3810 - val_accuracy: 0.7619 - val_loss: 0.6491
Epoch 2/5
13/13 - 2s - 177ms/step - accuracy: 0.7113 - loss: 0.8103 - val_accuracy: 0.8452 - val_loss: 0.4531
Epoch 3/5
13/13 - 2s - 178ms/step - accuracy: 0.8067 - loss: 0.5052 - val_accuracy: 0.9048 - val_loss: 0.2826
Epoch 4/5
13/13 - 2s - 180ms/step - accuracy: 0.8737 - loss: 0.3983 - val_accuracy: 0.9286 - val_loss: 0.2476
Epoch 5/5
13/13 - 2s - 180ms/step - accuracy: 0.8866 - loss: 0.3410 - val_accuracy: 0.9524 - val_loss: 0.2122
Epoch 1/25
13/13 - 25s - 2s/step - accuracy: 0.9227 - loss: 0.2761 - val_accuracy: 0.9643 - val_loss: 0.1033
Epoch 2/25
13/13 - 5s - 417ms/step - accuracy: 0.9253 - loss: 0.2373 - val_accuracy: 0.9762 - val_loss: 0.0578
Epoch 3/25
13/13 - 5s - 372ms/step - accuracy: 0.9227 - loss: 0.2099 - val_accuracy: 0.9643 - val_loss: 0.0842
Epoch 4/25
13/13 - 5s - 387ms/step - accuracy: 0.9691 - loss: 0.1236 - val_accuracy: 0.

In [16]:
# EfficientNet-B0
for seed in SEEDS:
    print(f"\n===== EfficientNet-B0 | seed {seed} =====")
    treinar("efficientnetb0", seed)
    keras.backend.clear_session()


===== EfficientNet-B0 | seed 0 =====
Epoch 1/5
13/13 - 13s - 963ms/step - accuracy: 0.4536 - loss: 1.1569 - val_accuracy: 0.7381 - val_loss: 0.9386
Epoch 2/5
13/13 - 1s - 97ms/step - accuracy: 0.7474 - loss: 0.8626 - val_accuracy: 0.8095 - val_loss: 0.7233
Epoch 3/5
13/13 - 1s - 97ms/step - accuracy: 0.7964 - loss: 0.7334 - val_accuracy: 0.8571 - val_loss: 0.6299
Epoch 4/5
13/13 - 1s - 97ms/step - accuracy: 0.8325 - loss: 0.6706 - val_accuracy: 0.8929 - val_loss: 0.5656
Epoch 5/5
13/13 - 1s - 97ms/step - accuracy: 0.8351 - loss: 0.6087 - val_accuracy: 0.8810 - val_loss: 0.5187
Epoch 1/25
13/13 - 25s - 2s/step - accuracy: 0.8428 - loss: 0.5662 - val_accuracy: 0.8810 - val_loss: 0.4581
Epoch 2/25
13/13 - 3s - 253ms/step - accuracy: 0.8840 - loss: 0.4842 - val_accuracy: 0.9048 - val_loss: 0.4077
Epoch 3/25
13/13 - 3s - 256ms/step - accuracy: 0.8918 - loss: 0.4206 - val_accuracy: 0.9167 - val_loss: 0.3726
Epoch 4/25
13/13 - 3s - 256ms/step - accuracy: 0.9227 - loss: 0.3690 - val_accuracy:

In [12]:
!ls -lh /content/drive/MyDrive/P2_runs/

total 953M
-rw------- 1 root root  48M Oct 26 18:41 efficientnetb0_seed0.keras
-rw------- 1 root root  48M Oct 26 18:43 efficientnetb0_seed1.keras
-rw------- 1 root root  48M Oct 26 18:45 efficientnetb0_seed2.keras
-rw------- 1 root root 8.9K Oct 26 18:42 preds_efficientnetb0_seed0_test.csv
-rw------- 1 root root 8.8K Oct 26 18:42 preds_efficientnetb0_seed0_val.csv
-rw------- 1 root root 8.9K Oct 26 18:44 preds_efficientnetb0_seed1_test.csv
-rw------- 1 root root 8.8K Oct 26 18:44 preds_efficientnetb0_seed1_val.csv
-rw------- 1 root root 8.9K Oct 26 18:45 preds_efficientnetb0_seed2_test.csv
-rw------- 1 root root 8.8K Oct 26 18:45 preds_efficientnetb0_seed2_val.csv
-rw------- 1 root root 8.8K Oct 26 18:33 preds_resnet50_seed0_test.csv
-rw------- 1 root root 8.7K Oct 26 18:33 preds_resnet50_seed0_val.csv
-rw------- 1 root root 8.7K Oct 26 18:35 preds_resnet50_seed1_test.csv
-rw------- 1 root root 8.6K Oct 26 18:35 preds_resnet50_seed1_val.csv
-rw------- 1 root root 8.8K Oct 26 18:39 pre

In [17]:
import glob
for f in sorted(glob.glob("/content/drive/MyDrive/P2_runs/preds_*_val.csv")):
    df = pd.read_csv(f)
    print(f"{f.split('/')[-1]:40s} accuracy val = {(df.y_true == df.y_pred).mean():.3f}")

preds_efficientnetb0_seed0_val.csv       accuracy val = 0.940
preds_efficientnetb0_seed1_val.csv       accuracy val = 0.917
preds_efficientnetb0_seed2_val.csv       accuracy val = 0.940
preds_resnet50_seed0_val.csv             accuracy val = 1.000
preds_resnet50_seed1_val.csv             accuracy val = 0.929
preds_resnet50_seed2_val.csv             accuracy val = 0.976
